## Week 1: Linear Regression - Polynomial Terms, Multicollinearity, 
## Interaction Terms, and Categorical Features

This notebook applies Week 1 concepts from DX799 Capstone Part 1 
to the BRFSS 2015 diabetes dataset. The goal is to explore linear 
regression techniques including polynomial terms, interaction terms, 
and multicollinearity detection using my capstone dataset.

Dataset: diabetes_binary_5050split_health_indicators_BRFSS2015.csv
Target variable: Diabetes_binary (0 = no diabetes, 1 = diabetes)

In [1]:
# DX799 Capstone Part 1 - Week 1
# Linear Regression: Polynomial Terms, Multicollinearity,
# Interaction Terms, Continuous and Categorical Features

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from statsmodels.stats.outliers_influence import variance_inflation_factor

## Dataset Overview

The balanced 50/50 binary dataset contains 70,692 rows and 22 columns.
The target variable Diabetes_binary is evenly split between 0 (no diabetes) 
and 1 (diabetes), making it a good dataset for exploration since we do not 
have to worry about class imbalance skewing our results.

In [4]:
df = pd.read_csv("diabetes_binary_5050split_health_indicators_BRFSS2015.csv")

In [6]:
print(df.shape)

(70692, 22)


In [7]:
print(df.head())

   Diabetes_binary  HighBP  HighChol  CholCheck   BMI  Smoker  Stroke  \
0              0.0     1.0       0.0        1.0  26.0     0.0     0.0   
1              0.0     1.0       1.0        1.0  26.0     1.0     1.0   
2              0.0     0.0       0.0        1.0  26.0     0.0     0.0   
3              0.0     1.0       1.0        1.0  28.0     1.0     0.0   
4              0.0     0.0       0.0        1.0  29.0     1.0     0.0   

   HeartDiseaseorAttack  PhysActivity  Fruits  ...  AnyHealthcare  \
0                   0.0           1.0     0.0  ...            1.0   
1                   0.0           0.0     1.0  ...            1.0   
2                   0.0           1.0     1.0  ...            1.0   
3                   0.0           1.0     1.0  ...            1.0   
4                   0.0           1.0     1.0  ...            1.0   

   NoDocbcCost  GenHlth  MentHlth  PhysHlth  DiffWalk  Sex   Age  Education  \
0          0.0      3.0       5.0      30.0       0.0  1.0   4.0   

## Features and Target

21 features will be used as predictors. All features are already 
numeric. Most categorical features like HighBP, Smoker, and 
PhysActivity are already one-hot encoded as 0 or 1, so no 
additional encoding is needed for this dataset.

Top positive correlations with diabetes: GenHlth, HighBP, BMI, 
HighChol, Age, DiffWalk

Top negative correlations with diabetes: Income, Education, 
PhysActivity

These will be the focus features for polynomial and interaction 
term exploration this week.

In [8]:
target = 'Diabetes_binary'

feature_cols = [col for col in df.columns if col != target]

X = df[feature_cols]
y = df[target]

print("Features:", feature_cols)
print("Target:", target)
print("X shape:", X.shape)
print("y shape:", y.shape)

Features: ['HighBP', 'HighChol', 'CholCheck', 'BMI', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'GenHlth', 'MentHlth', 'PhysHlth', 'DiffWalk', 'Sex', 'Age', 'Education', 'Income']
Target: Diabetes_binary
X shape: (70692, 21)
y shape: (70692,)


## Correlation Analysis

Before adding any polynomial or interaction terms, I want to identify 
which features have the strongest relationship with Diabetes_binary. 
This will help me focus on the most meaningful candidates rather than 
adding complexity to features that barely relate to the target.

Features with correlation above 0.25 or below -0.15 will be 
prioritized for further exploration.

In [9]:
corr = df.corr()
target_corr = corr['Diabetes_binary'].drop('Diabetes_binary').sort_values(ascending=False)
print(target_corr)

GenHlth                 0.407612
HighBP                  0.381516
BMI                     0.293373
HighChol                0.289213
Age                     0.278738
DiffWalk                0.272646
PhysHlth                0.213081
HeartDiseaseorAttack    0.211523
Stroke                  0.125427
CholCheck               0.115382
MentHlth                0.087029
Smoker                  0.085999
Sex                     0.044413
NoDocbcCost             0.040977
AnyHealthcare           0.023191
Fruits                 -0.054077
Veggies                -0.079293
HvyAlcoholConsump      -0.094853
PhysActivity           -0.158666
Education              -0.170481
Income                 -0.224449
Name: Diabetes_binary, dtype: float64


## Baseline Linear Regression

Running a baseline OLS linear regression using all 21 features before 
adding any polynomial or interaction terms. This gives us a reference 
point to compare against as we add complexity to the model.

Note: Diabetes_binary is a categorical target (0 or 1), which means 
linear regression is not the ideal tool for this dataset. Logistic 
regression (Week 4) will be more appropriate. However, the course 
asks us to apply linear regression as part of the exploration process 
this week.

In [10]:
# Baseline linear regression - statsmodels
X_const = sm.add_constant(X)
baseline_model = sm.OLS(y, X_const)
baseline_results = baseline_model.fit()
print(baseline_results.summary())

                            OLS Regression Results                            
Dep. Variable:        Diabetes_binary   R-squared:                       0.309
Model:                            OLS   Adj. R-squared:                  0.309
Method:                 Least Squares   F-statistic:                     1508.
Date:                Thu, 14 May 2026   Prob (F-statistic):               0.00
Time:                        20:23:25   Log-Likelihood:                -38218.
No. Observations:               70692   AIC:                         7.648e+04
Df Residuals:                   70670   BIC:                         7.668e+04
Df Model:                          21                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   -0.5861 

## Multicollinearity Check - Baseline VIF

Before adding polynomial terms, checking for multicollinearity in 
the baseline feature set using Variance Inflation Factor (VIF). 
A VIF above 5 is a warning sign. A VIF above 10 is a serious problem.

Running this now so we have a clean baseline to compare against 
after polynomial terms are added.

In [12]:
# Check VIF for all features
X_const = sm.add_constant(X)

vif_data = pd.DataFrame()
vif_data['feature'] = X_const.columns
vif_data['VIF'] = [variance_inflation_factor(X_const.values, i)
                   for i in range(len(X_const.columns))]

print(vif_data.sort_values('VIF', ascending=False))

                 feature         VIF
0                  const  126.858651
14               GenHlth    1.890908
16              PhysHlth    1.700399
17              DiffWalk    1.591225
21                Income    1.544598
1                 HighBP    1.357474
19                   Age    1.337148
20             Education    1.331728
15              MentHlth    1.267156
7   HeartDiseaseorAttack    1.193935
4                    BMI    1.180737
2               HighChol    1.179017
8           PhysActivity    1.167430
13           NoDocbcCost    1.143393
10               Veggies    1.103377
9                 Fruits    1.101333
12         AnyHealthcare    1.095602
6                 Stroke    1.094027
18                   Sex    1.088941
5                 Smoker    1.082180
3              CholCheck    1.032894
11     HvyAlcoholConsump    1.022839


## Polynomial Terms

Based on the correlation analysis, adding squared terms for the top 
positive and negative predictors: GenHlth, BMI, Age, DiffWalk, 
and Income. These are the features most strongly related to the 
target and most likely to have non-linear relationships with 
diabetes risk.

In [14]:
# Add polynomial (squared) terms for top features
poly_features = ['GenHlth', 'BMI', 'Age', 'DiffWalk', 'Income']

X_poly = X.copy()
for feature in poly_features:
    X_poly[f'{feature}2'] = X_poly[feature] ** 2

print("Original shape:", X.shape)
print("New shape with polynomial terms:", X_poly.shape)
print("Columns added:", [f'{f}2' for f in poly_features])

Original shape: (70692, 21)
New shape with polynomial terms: (70692, 26)
Columns added: ['GenHlth2', 'BMI2', 'Age2', 'DiffWalk2', 'Income2']


## Linear Regression with Polynomial Terms

Running OLS regression with the five squared terms added. 
Comparing R-squared to the baseline (0.309) to see whether 
the polynomial terms improved the model's ability to explain 
variance in diabetes outcomes.

In [15]:
# Run regression with polynomial terms
X_poly_const = sm.add_constant(X_poly)
poly_model = sm.OLS(y, X_poly_const)
poly_results = poly_model.fit()

print(f"Baseline R-squared:         0.309")
print(f"Polynomial R-squared:       {poly_results.rsquared:.3f}")
print(f"Difference:                 {poly_results.rsquared - 0.309:.3f}")

Baseline R-squared:         0.309
Polynomial R-squared:       0.317
Difference:                 0.008


## Multicollinearity Check - After Polynomial Terms

Re-running VIF on the expanded feature set to check whether 
the squared terms created any multicollinearity issues. 
Comparing against the baseline VIF where all features were below 2.0.

In [16]:
# Re-check VIF after adding polynomial terms
X_poly_const = sm.add_constant(X_poly)

vif_poly = pd.DataFrame()
vif_poly['feature'] = X_poly_const.columns
vif_poly['VIF'] = [variance_inflation_factor(X_poly_const.values, i)
                   for i in range(len(X_poly_const.columns))]

print(vif_poly.sort_values('VIF', ascending=False))

/opt/anaconda3/lib/python3.12/site-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)


                 feature         VIF
17              DiffWalk         inf
25             DiffWalk2         inf
0                  const  265.390541
26               Income2   27.649340
21                Income   27.352312
22              GenHlth2   26.324418
14               GenHlth   24.433907
24                  Age2   23.020953
19                   Age   22.917754
4                    BMI   18.990672
23                  BMI2   18.237031
16              PhysHlth    1.854149
1                 HighBP    1.373535
20             Education    1.337382
15              MentHlth    1.275651
7   HeartDiseaseorAttack    1.205549
2               HighChol    1.192368
8           PhysActivity    1.169204
13           NoDocbcCost    1.147275
9                 Fruits    1.103961
10               Veggies    1.103680
12         AnyHealthcare    1.097124
6                 Stroke    1.096110
18                   Sex    1.095060
5                 Smoker    1.086967
3              CholCheck    1.033254
1

## Fixing Multicollinearity

DiffWalk is binary (0 or 1) so squaring it produces an identical 
column, causing infinite VIF. Removing DiffWalk2. Also removing 
squared terms for Income, GenHlth, Age, and BMI due to high VIF 
scores. Will keep only squared terms that do not cause problems.

In [17]:
# Remove problematic polynomial terms
X_poly_fixed = X_poly.drop(columns=['DiffWalk2', 'Income2', 
                                     'GenHlth2', 'Age2', 'BMI2'])

print("Remaining columns:", X_poly_fixed.shape)

Remaining columns: (70692, 21)


## Polynomial Terms Finding

After checking VIF post-polynomial addition, all five squared terms 
caused significant multicollinearity (VIF well above 5). DiffWalk 
is binary (0 or 1) so squaring it produces an identical column. 
The other features (Income, GenHlth, Age, BMI) have limited ranges 
that cause X and X-squared to move nearly in lockstep.

This suggests the BRFSS dataset features do not have wide enough 
continuous ranges to benefit from polynomial terms without introducing 
multicollinearity. The baseline linear model with original features 
is more stable for this dataset. Regularization techniques like 
Lasso (Week 2) may be a better approach to improving the model.

## Interaction Terms

Testing an interaction term between BMI and Age. The hypothesis is 
that the effect of BMI on diabetes risk is stronger for older 
individuals than younger ones. A 65-year-old with high BMI has 
had more years of metabolic stress than a 25-year-old with the 
same BMI, so the combination may compound diabetes risk more than 
either feature alone.

Adding BMI * Age as a new feature and comparing R-squared to baseline.

In [18]:
# Add interaction term: BMI x Age
X_interact = X.copy()
X_interact['BMI_x_Age'] = X_interact['BMI'] * X_interact['Age']

# Run regression with interaction term
X_interact_const = sm.add_constant(X_interact)
interact_model = sm.OLS(y, X_interact_const)
interact_results = interact_model.fit()

print(f"Baseline R-squared:           0.309")
print(f"Interaction R-squared:        {interact_results.rsquared:.3f}")
print(f"Difference:                   {interact_results.rsquared - 0.309:.3f}")
print(f"\nBMI_x_Age coefficient:        {interact_results.params['BMI_x_Age']:.6f}")
print(f"BMI_x_Age p-value:            {interact_results.pvalues['BMI_x_Age']:.6f}")

Baseline R-squared:           0.309
Interaction R-squared:        0.311
Difference:                   0.002

BMI_x_Age coefficient:        0.001020
BMI_x_Age p-value:            0.000000


## Interaction Term Finding

The BMI x Age interaction term is statistically significant 
(p-value essentially 0) confirming that the effect of BMI on 
diabetes risk does depend on age. Older individuals with high 
BMI carry compounded risk. However the R-squared improvement 
is modest (0.002), suggesting the interaction adds real but 
limited explanatory power beyond the individual features alone.

## Model Comparison Summary

Comparing R-squared across all three models: baseline, polynomial, 
and interaction. This gives a clear picture of how each addition 
affected the model's explanatory power.

In [20]:
# Summary comparison of all three models
print("=" * 50)
print("MODEL COMPARISON SUMMARY")
print("=" * 50)
print(f"Baseline (21 features):          R² = 0.309")
print(f"Polynomial terms (removed):      R² = 0.309")
print(f"  (all polynomial terms caused multicollinearity)")
print(f"Interaction BMI x Age:           R² = {interact_results.rsquared:.3f}")
print("=" * 50)
print(f"\nKey findings:")
print(f"1. Baseline VIF: all features below 2.0 - clean")
print(f"2. Polynomial terms: caused high VIF across")
print(f"   all candidates - removed")
print(f"3. BMI x Age interaction: significant (p<0.001)")
print(f"   Small but real improvement in R²")
print(f"4. Linear regression is not ideal for this")
print(f"   dataset - target is categorical (0 or 1)")
print(f"   Logistic regression (Week 4) will fit better")

MODEL COMPARISON SUMMARY
Baseline (21 features):          R² = 0.309
Polynomial terms (removed):      R² = 0.309
  (all polynomial terms caused multicollinearity)
Interaction BMI x Age:           R² = 0.311

Key findings:
1. Baseline VIF: all features below 2.0 - clean
2. Polynomial terms: caused high VIF across
   all candidates - removed
3. BMI x Age interaction: significant (p<0.001)
   Small but real improvement in R²
4. Linear regression is not ideal for this
   dataset - target is categorical (0 or 1)
   Logistic regression (Week 4) will fit better
